# Label lab: specimen label table, fast loop

Reads the label fields (ID, species, age, sex, feather line, date) from the **original** slides and
shows exactly what the model saw. Loop: change a parameter or `configs/label_fields.json` →
re-run from **2. Run** → look at the table and the review gallery → repeat. A 10-image sample takes
about a minute.

Each field tries its primary crop, an expanded crop, then wider fallback crops (`fallback_regions`).
Field statuses: `readable`, `partial` (some text hidden), `not_found` (absent in every crop; exported
`N/A`), `needs_review` (bad format, or two readings disagree), `conflict` (disagrees with the ID/year in
the file name), `filename_fallback` (nothing on the slide; value from the file name), `model_error`
(request failed; **not** the same as missing, exported blank). `file_id` is the file name;
`individual_id` is the code printed on the label.

The model runs on the VLM server (`pixi run vlm-server` on an Apple-silicon node; set
`FEATHER_VLM_ENDPOINTS` in `.env`); nothing heavy runs here.
Kernel: **bird-feather-analysis (pixi)**.

## 1. Parameters

In [ ]:
SAMPLE = 10            # random images to run (ignored when MATCH is set)
SEED = 1               # change for a different random sample
MATCH = []             # e.g. ["A0012 1996", "BB17692"] to run specific slides
FIELDS = None          # e.g. ["date", "individual_id"] to iterate on a few fields; None = all
WORKERS = 3            # images in flight at once
CONFIG = "../configs/label_fields.json"
GOLD = "../data/label_gold.csv"   # your hand-checked answers (created in step 5)

In [ ]:
import json, random, sys, time
from pathlib import Path
sys.path.insert(0, "..")
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from IPython.display import display
from PIL import ImageDraw
from src import label_extract as lx
from src.notebook_remote_runtime import RemoteClusterIO, load_notebook_config_from_env

rc = load_notebook_config_from_env()
remote = RemoteClusterIO(rc)
CACHE = Path("../data/raw_cache"); CACHE.mkdir(parents=True, exist_ok=True)
ALL_IMAGES = remote.list_remote_images()
print(len(ALL_IMAGES), "slides on the head node")

def local_copy(remote_path):
    p = CACHE / Path(remote_path).name
    if not p.exists():
        remote.fetch_remote_file(rc.head_host, remote_path, CACHE)
    return p

## 2. Run

In [ ]:
cfg = lx.load_config(CONFIG)          # re-read every run, so config edits apply
client = lx.VLMClient(cfg["vlm"])
if MATCH:
    chosen = [p for p in ALL_IMAGES if any(m in Path(p).name for m in MATCH)]
else:
    chosen = random.Random(SEED).sample(ALL_IMAGES, min(SAMPLE, len(ALL_IMAGES)))
t0 = time.time()
with ThreadPoolExecutor(WORKERS) as pool:
    results = list(pool.map(lambda p: lx.extract(local_copy(p), cfg, client, FIELDS), chosen))
df = pd.DataFrame([r.row() for r in results])
print(f"{len(results)} slides in {time.time()-t0:.0f}s, {int(df.needs_review.sum())} need review")

STATUS_COLORS = {"readable": "#d4edda", "partial": "#fff3cd", "not_found": "#e2e3e5",
                 "filename_fallback": "#fff3cd", "needs_review": "#f8d7da", "conflict": "#f8d7da",
                 "model_error": "#f5c6cb"}
def _color(col):
    st = df.get(col + "_status")
    return [f"background-color: {STATUS_COLORS.get(s, '')}" for s in st] if st is not None else [""] * len(df)
value_cols = ["file_id", "needs_review"] + [f for f in lx.FIELDS_ORDER if f in df]
display(df[value_cols].style.apply(lambda c: _color(c.name) if c.name in lx.FIELDS_ORDER else [""] * len(c)))

## 3. Review gallery
Every field that needs review, with the exact crop the model read and its raw answer.

In [ ]:
for r in results:
    bad = {n: f for n, f in r.fields.items() if f.status not in ("readable", "not_found")}
    if not bad:
        continue
    crops = lx.field_crops(r.image, cfg)
    print("=" * 100); print(Path(r.image).name)
    for n, f in bad.items():
        print(f"  {n}: {f.value!r}  [{f.status}] via {f.source or '-'}  {f.note}  raw={f.raw[:160]!r}")
        im = crops[n].copy(); im.thumbnail((700, 220)); display(im)

## 4. Tune regions
Draws every field's region on a slide. Edit `region` in `configs/label_fields.json`, then re-run this cell.

In [ ]:
SLIDE = chosen[0] if chosen else ALL_IMAGES[0]
cfg = lx.load_config(CONFIG)
from PIL import Image
im = Image.open(local_copy(SLIDE)).convert("RGB"); w, h = im.size
d = ImageDraw.Draw(im)
for n, s in cfg["fields"].items():
    l, t, rr, b = s["region"]
    d.rectangle([l*w, t*h, rr*w, b*h], outline="red", width=8); d.text((l*w+12, t*h+12), n, fill="red")
im.thumbnail((900, 900)); display(im)

## 5. Gold answers and accuracy
Correct the values for a handful of slides once, and every later run is scored against them.
The first run writes `data/label_gold.csv` from the current results; open it in JupyterLab, fix the
wrong cells, save, and re-run this cell.

In [ ]:
gold = Path(GOLD)
if not gold.exists():
    df[["image"] + [f for f in lx.FIELDS_ORDER if f in df]].to_csv(gold, index=False)
    print("wrote", gold, "- correct it by hand, then re-run this cell")
else:
    for name, s in lx.score_against_gold(results, gold).items():
        print(f"{name:14} {s['correct']:>3}/{s['n']:<3} accuracy {s['accuracy']}")
        for img, got, want, st in s["wrong"][:5]:
            print(f"      {img}: got {got!r} want {want!r} [{st}]")

## 6. Save this run

In [ ]:
out = lx.write_csv(results, Path("../data/label_runs") / time.strftime("%Y%m%d-%H%M%S") / "labels.csv")
(out.parent / "config_used.json").write_text(json.dumps(cfg, indent=2))
print(out)
# Full table for all slides (long: ~15 s per slide per model server):
#   pixi run labels-all